In [1]:
import sys
import os 
sys.path.append(os.path.abspath(".."))
import pandas as pd
from src.preprocessing import *
from src.features import *
import numpy as np

In [38]:
df = pd.read_csv("../data/raw/raw_matches.csv")

df.head()

,match_id,radiant_win,start_time,duration,leagueid,radiant_score,dire_score,radiant_team_id,radiant_team_name,dire_team_id,dire_team_name,series_id,series_type
0,8712056091,True,1772388652,2833,19269,32,31,8291895,NaN,9467224,NaN,1069888.0,2.0
1,8711970053,True,1772384823,1919,19269,37,8,9467224,NaN,8291895,NaN,1069888.0,2.0
2,8711885801,False,1772381334,1691,19269,6,28,9467224,NaN,8291895,NaN,1069888.0,2.0
3,8711768658,False,1772376899,2508,19269,11,37,9467224,NaN,8291895,NaN,1069888.0,2.0
4,8711578057,True,1772370355,2080,19269,30,15,9467224,NaN,2163,NaN,1069815.0,1.0


In [39]:
df.isna().sum()

match_id               0
radiant_win            0
start_time             0
duration               0
leagueid               0
radiant_score          0
dire_score             0
radiant_team_id        0
radiant_team_name    933
dire_team_id           0
dire_team_name       933
series_id              2
series_type            2
dtype: int64

In [40]:
df.duplicated().sum()

0

In [41]:
df = preprocess_matches(df)
df = add_game_features(df)

In [42]:
df.to_csv("../data/processed/clean_matches.csv", index=False)

In [43]:
cols_flat = ["match_id", "throw", "comeback", "loss", "first_blood_time", "patch"]
match_detail_df[cols_flat].to_csv("../data/raw/match_detail_flat.csv", index=False)

In [2]:
clean_matches_df = pd.read_csv("../data/processed/clean_matches.csv")
team_mapping_df = pd.read_csv("../data/processed/team_mapping.csv")
match_detail_flat_df = pd.read_csv("../data/raw/match_detail_flat.csv")


In [3]:
clean_matches_df = clean_matches_df.drop(columns=["radiant_team_name", "dire_team_name"])


clean_matches_df = clean_matches_df.merge(team_mapping_df, left_on="radiant_team_id", right_on="team_id", how="left")
clean_matches_df = clean_matches_df.rename(columns={"team_name": "radiant_team_name"})
clean_matches_df = clean_matches_df.drop(columns=["team_id"])

clean_matches_df = clean_matches_df.merge(team_mapping_df, left_on="dire_team_id", right_on="team_id", how="left")
clean_matches_df = clean_matches_df.rename(columns={"team_name": "dire_team_name"})
clean_matches_df = clean_matches_df.drop(columns=["team_id"])

In [72]:
clean_matches_df

,match_id,radiant_win,start_time,duration,leagueid,radiant_score,dire_score,radiant_team_id,dire_team_id,series_id,series_type,total_kills,game_speed_categorize,radiant_team_name,dire_team_name
0,8712056091,True,2026-03-01 18:10:52,47.22,19269,32,31,8291895,9467224,1069888.0,2.0,63,Late,Tundra Esports,Aurora Gaming
1,8711970053,True,2026-03-01 17:07:03,31.98,19269,37,8,9467224,8291895,1069888.0,2.0,45,Normal,Aurora Gaming,Tundra Esports
2,8711885801,False,2026-03-01 16:08:54,28.18,19269,6,28,9467224,8291895,1069888.0,2.0,34,Fast,Aurora Gaming,Tundra Esports
3,8711768658,False,2026-03-01 14:54:59,41.80,19269,11,37,9467224,8291895,1069888.0,2.0,48,Late,Aurora Gaming,Tundra Esports
4,8711578057,True,2026-03-01 13:05:55,34.67,19269,30,15,9467224,2163,1069815.0,1.0,45,Normal,Aurora Gaming,Team Liquid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
928,8675882674,False,2026-02-03 11:17:05,40.17,19099,18,39,2163,2586976,1061169.0,0.0,57,Late,Team Liquid,OG
929,8675848583,False,2026-02-03 10:41:00,32.45,19099,6,23,9823272,9303484,1061166.0,0.0,29,Normal,Team Yandex,HEROIC
930,8675821146,True,2026-02-03 10:09:07,42.37,19099,32,19,7119388,36,1061164.0,0.0,51,Late,Team Spirit,Natus Vincere
931,8675768384,True,2026-02-03 09:00:34,72.28,19099,37,34,8291895,9828897,1061159.0,0.0,71,Late,Tundra Esports,REKONIX


In [4]:
clean_matches_df.to_csv("../data/processed/clean_matches.csv", index=False)

In [6]:
throw_comeback_df = clean_matches_df.merge(match_detail_flat_df, on="match_id", how="left")

In [8]:
throw_comeback_df[["match_id", "throw", "comeback"]].head(10)

,match_id,throw,comeback
0,8712056091,7136.0,NaN
1,8711970053,0.0,NaN
2,8711885801,NaN,57.0
3,8711768658,NaN,326.0
4,8711578057,1828.0,NaN
5,8711465051,370.0,NaN
6,8710481024,1029.0,NaN
7,8710371476,NaN,743.0
8,8710260672,NaN,392.0
9,8710135523,NaN,100.0


In [9]:
throw_comeback_df[["throw", "comeback"]].notna().sum()

throw       461
comeback    472
dtype: int64

In [10]:
throw_comeback_df[["throw", "comeback"]].describe()

,throw,comeback
count,461.000000,472.000000
mean,2543.832972,2163.951271
std,4137.707510,3404.751669
min,-920.000000,-899.000000
25%,155.000000,62.250000
50%,914.000000,762.000000
75%,3045.000000,2791.000000
max,28310.000000,21195.000000


In [11]:
print(throw_comeback_df[throw_comeback_df["throw"] < 0][["match_id", "throw", "comeback"]])

       match_id  throw  comeback
66   8702081221  -17.0       NaN
184  8694692688 -700.0       NaN
276  8562561579   -2.0       NaN
295  8560525197  -21.0       NaN
317  8549890849 -504.0       NaN
341  8518560731  -12.0       NaN
385  8512795885  -21.0       NaN
409  8594217096 -658.0       NaN
434  8583851046 -920.0       NaN
447  8582157073  -30.0       NaN
601  8603728193  -11.0       NaN
635  8601850047 -525.0       NaN
666  8600397564 -508.0       NaN
828  8524344903 -100.0       NaN


In [16]:
print(throw_comeback_df[["radiant_team_name", "dire_team_name"]].isna().sum())

radiant_team_name    0
dire_team_name       0
dtype: int64


In [19]:
throw_comeback_df["throw_team"] = np.where(throw_comeback_df["radiant_win"] == True, throw_comeback_df["dire_team_name"], throw_comeback_df["radiant_team_name"])
throw_comeback_df["comeback_team"] = np.where(throw_comeback_df["radiant_win"] == True, throw_comeback_df["radiant_team_name"], throw_comeback_df["dire_team_name"])

In [20]:
throw_comeback_df.head(5)

,match_id,radiant_win,start_time,duration,leagueid,radiant_score,dire_score,radiant_team_id,dire_team_id,series_id,...,game_speed_categorize,radiant_team_name,dire_team_name,throw,comeback,loss,first_blood_time,patch,throw_team,comeback_team
0,8712056091,True,2026-03-01 18:10:52,47.22,19269,32,31,8291895,9467224,1069888.0,...,Late,Tundra Esports,Aurora Gaming,7136.0,NaN,9262.0,32,59,Aurora Gaming,Tundra Esports
1,8711970053,True,2026-03-01 17:07:03,31.98,19269,37,8,9467224,8291895,1069888.0,...,Normal,Aurora Gaming,Tundra Esports,0.0,NaN,13548.0,142,59,Tundra Esports,Aurora Gaming
2,8711885801,False,2026-03-01 16:08:54,28.18,19269,6,28,9467224,8291895,1069888.0,...,Fast,Aurora Gaming,Tundra Esports,NaN,57.0,NaN,50,59,Aurora Gaming,Tundra Esports
3,8711768658,False,2026-03-01 14:54:59,41.80,19269,11,37,9467224,8291895,1069888.0,...,Late,Aurora Gaming,Tundra Esports,NaN,326.0,NaN,1,59,Aurora Gaming,Tundra Esports
4,8711578057,True,2026-03-01 13:05:55,34.67,19269,30,15,9467224,2163,1069815.0,...,Normal,Aurora Gaming,Team Liquid,1828.0,NaN,18091.0,71,59,Team Liquid,Aurora Gaming


In [21]:
throw_comeback_df.to_csv("../data/processed/comeback_throw_team.csv", index=False)